In [1]:
import pandas as pd
import numpy as np
from textblob import TextBlob
from scipy import stats
import warnings

# Suppress minor pandas warnings to keep the output clean
warnings.filterwarnings('ignore')

# Step 1 - Preprocessing

In [2]:
# Load the dataset
df = pd.read_csv('synthetic_social_media_engagement.csv')

# Convert date strings to actual datetime objects
df['post_date'] = pd.to_datetime(df['post_date'])
df['account_creation_date'] = pd.to_datetime(df['account_creation_date'])

# Filter for English language posts to ensure NLP model accuracy
df = df[df['language'] == 'en'].copy()

# Extract month and day of the week for later trend analysis
df['post_month'] = df['post_date'].dt.to_period('M')
df['post_day_of_week'] = df['post_date'].dt.day_name()

print(f"Data shape after cleaning: {df.shape}")

Data shape after cleaning: (4006, 24)


# Step 2 - Feature Engineering

In [5]:
# Prevent division by zero by replacing 0 followers with 1
safe_followers = df['followers_count'].replace(0, 1)

# Calculate Viral Score (heavily weighting shares and comments over likes)
df['viral_score'] = ((df['shares'] * 3.0) + (df['comments'] * 1.5) + (df['likes'] * 0.5)) / safe_followers

# Group content into actionable formats for A/B testing
def categorize_content(row):
    if row['has_media'] and row['content_length'] > 120:
        return 'Long Media'
    elif row['has_media']:
        return 'Short Media'
    elif row['content_length'] > 120:
        return 'Long Text'
    else:
        return 'Short Text'

df['content_format'] = df.apply(categorize_content, axis=1)

# Step 3 - NLP Sentiment & Relatability

In [6]:
# Calculate sentiment polarity (-1.0 to 1.0)
df['sentiment_polarity'] = df['post_content'].apply(lambda text: TextBlob(str(text)).sentiment.polarity)

# Keywords indicating vulnerability or problem-awareness
relatable_keywords = ['i feel', 'struggle', 'anyone else', 'hard time', 'relatable', 'me too', 'anxiety', 'worry']

def tag_relatability(text, v_score, avg_v_score):
    text_lower = str(text).lower()
    has_keyword = any(kw in text_lower for kw in relatable_keywords)

    # It's only truly relatable if it includes problem-aware language AND performs above average
    if has_keyword and v_score > avg_v_score:
        return 'High'
    return 'Neutral'

# Apply the tagging function
avg_viral = df['viral_score'].mean()
df['relatability_class'] = df.apply(
    lambda row: tag_relatability(row['post_content'], row['viral_score'], avg_viral),
    axis=1
)

print(df['relatability_class'].value_counts())

relatability_class
Neutral    3959
High         47
Name: count, dtype: int64


# Step 4 - Statistical Testing Framework

In [7]:
# A/B Test 1: Does having media significantly impact the Viral Score?
with_media = df[df['has_media'] == True]['viral_score']
without_media = df[df['has_media'] == False]['viral_score']

t_stat, p_val_media = stats.ttest_ind(with_media, without_media, equal_var=False)
print(f"Media vs No Media p-value: {p_val_media:.5f}")

if p_val_media < 0.05:
    print("Result: Including media causes a statistically significant difference in virality.")
else:
    print("Result: Media presence does not significantly impact virality in this dataset.")

# A/B Test 2: Does the Topic change the amount of shares? (ANOVA test)
topics = df['topic'].unique()
topic_groups = [df[df['topic'] == t]['shares'] for t in topics]

f_stat, p_val_topic = stats.f_oneway(*topic_groups)
print(f"\nTopic differences p-value: {p_val_topic:.5f}")

Media vs No Media p-value: 0.60953
Result: Media presence does not significantly impact virality in this dataset.

Topic differences p-value: 0.81144


# Step 5 - Recommender Engine & Export

In [8]:
# Engagement Optimization Recommender
def recommend_strategy(target_topic):
    topic_data = df[df['topic'] == target_topic]
    if len(topic_data) == 0:
        return "Topic not found."

    # Find which format got the highest average viral score for this topic
    best_format = topic_data.groupby('content_format')['viral_score'].mean().idxmax()
    avg_length = int(topic_data['content_length'].mean())

    print(f"--- Strategy for {target_topic} ---")
    print(f"Recommended Format: {best_format}")
    print(f"Suggested Text Length: ~{avg_length} characters")

# Test the recommender with a sample topic from the dataset
recommend_strategy('Technology')

# Export the final processed dataframe to use in PowerBI/Streamlit
df.to_csv('cleaned_and_modeled_data.csv', index=False)
print("\nExport complete: 'cleaned_and_modeled_data.csv' is ready for the dashboard.")

--- Strategy for Technology ---
Recommended Format: Short Text
Suggested Text Length: ~125 characters

Export complete: 'cleaned_and_modeled_data.csv' is ready for the dashboard.
